# Train Your Own YOLO Face Mask Detector

Run this notebook in **Google Colab** with a free GPU (`Runtime > Change runtime type > T4 GPU`).

It will:
1. Install Ultralytics (YOLOv11) and Roboflow
2. Download a labeled face-mask dataset (with_mask / without_mask / mask_weared_incorrect)
3. Fine-tune a YOLO model on it
4. Validate it and show you the mAP scores
5. Let you download `best.pt` to drop into the `models/` folder of the local project

Training on ~850-1500 images for 100 epochs on a free T4 GPU takes roughly **20-40 minutes**.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Get the dataset

You need a **free** Roboflow account + API key (universe.roboflow.com → sign up → Settings → API Key). Roboflow hosts many ready-made, already-annotated face-mask datasets exported directly in YOLOv11 format.

Good public datasets to pick from (browse and pick whichever looks best to you, then fill in the cell below):
- `https://universe.roboflow.com/print/face-mask-detection-aawx3` (based on the well-known Kaggle "Face Mask Detection" set — 3 classes: with_mask / without_mask / mask_weared_incorrect)
- `https://universe.roboflow.com/data-jrain/face-mask-detection-hdvgg` (1,424 images)
- Or search "face mask detection" on https://universe.roboflow.com/search and pick any dataset with a large image count and clean labels.

Open your chosen project page → **Dataset** tab → **Download this Dataset** → YOLO format supported by your installed Ultralytics version → it will show you the exact `workspace`, `project` and `version` values to paste below (or just copy the whole download code snippet it gives you and replace the cell below with it).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"   # <-- paste your key
WORKSPACE = "print"                       # <-- from the dataset page URL
PROJECT = "face-mask-detection-aawx3"     # <-- from the dataset page URL
VERSION = 1                                # <-- version number shown on the page

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov11")
print("Dataset downloaded to:", dataset.location)

## 2. Train YOLOv11

`yolo11n.pt` (nano) is a lightweight Ultralytics model and is what's used below — good enough for a real-time demo. Swap to `yolo11s.pt` for a bit more accuracy at the cost of speed if you have time.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name="face_mask_yolov11",
)

## 3. Validate — check your mAP@50 and mAP@50-95

In [ ]:
metrics = model.val()
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 4. Quick sanity-check prediction on a validation image

In [ ]:
import glob, random
from IPython.display import Image, display

val_images = glob.glob(f"{dataset.location}/valid/images/*")
sample = random.choice(val_images)

pred = model.predict(sample, conf=0.4, save=True)
print("Saved annotated prediction to:", pred[0].save_dir)
display(Image(filename=f"{pred[0].save_dir}/{sample.split('/')[-1]}"))

## 5. Download your trained weights

This grabs `best.pt` from the training run. Save it, then on your own machine place it at:

```
face_mask_detection/models/mask_detector.pt
```

and run `python main.py` as usual — it will use your freshly trained model.

In [ ]:
from google.colab import files

best_path = f"runs/detect/face_mask_yolov11/weights/best.pt"
print("Downloading:", best_path)
files.download(best_path)